In [1]:
from neo4j import GraphDatabase, RoutingControl

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "password")

In [2]:
def add_friend(driver, name, friend_name):
    driver.execute_query(
    "MERGE (a:Person {name: $name}) "
    "MERGE (friend:Person {name: $friend_name}) "
    "MERGE (a)-[:KNOWS]->(friend)",
    name=name, friend_name=friend_name, database_= "neo4j",
    )

In [3]:
def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_= "neo4j", routing_= RoutingControl.READ
    )
    for record in records:
        print(record["friend.name"])

with GraphDatabase.driver(URI, auth=AUTH, ) as driver:
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")


Guinevere
Lancelot
Merlin


# Check Points quetion


1. 
> 
2. 
>   
3. 
> 

# 1.3 Melakukan proses Impor Data dari RetailRocket ke Database Neo4J

In [4]:
from neo4j import GraphDatabase
import pandas as pd
from tqdm import tqdm
import math

driver = GraphDatabase.driver(URI, auth=AUTH)
session = driver.session(database="system")

csv_file_path = "../UTS/Data/events.csv"  

In [5]:
def import_batch(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid}) 
    WITH v, i, row
    CALL {
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)
    UNION ALL

        WITH v, i, row
        WITH v, i, row WHERE row.event = 'addtocart'

    CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)
    
    UNION ALL
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'transaction'

    CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)

In [6]:
batch_size = 5000
total_rows = 2000000  # ambil sebagian 


def create_constraints(session):
    session.run(
        "CREATE CONSTRAINT visitor_id_unique IF NOT EXISTS "
        "FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE"
    ).consume()

    session.run(
        "CREATE CONSTRAINT item_id_unique IF NOT EXISTS "
        "FOR (i:Item) REQUIRE i.itemId IS UNIQUE"
    ).consume()

    session.run(
        "CREATE CONSTRAINT transaction_id_unique IF NOT EXISTS "
        "FOR (t:Transaction) REQUIRE t.transactionId IS UNIQUE"
    ).consume()

    # Only useful if you import category_tree.csv; delete this block otherwise
    session.run(
        "CREATE CONSTRAINT category_id_unique IF NOT EXISTS "
        "FOR (c:Category) REQUIRE c.categoryId IS UNIQUE"
    ).consume()


def run_import(driver, csv_file_path):
    with driver.session(database="neo4j") as session:
        create_constraints(session)          # must run BEFORE the import

        print(f"Membaca file data dari: {csv_file_path}")
        chunks = pd.read_csv(
            csv_file_path,
            chunksize=batch_size,
            nrows=total_rows,                 # actually stops at 2M rows
            dtype={"transactionid": "string"},  # avoid "123.0"
        )
        total_chunks = math.ceil(total_rows / batch_size)

        print("Memulai proses impor data ke Neo4j...")
        for chunk in tqdm(chunks, total=total_chunks, desc="Mengimpor Batch Data"):
            # Membersihkan data null/NaN pada transactionid
            chunk["transactionid"] = chunk["transactionid"].fillna("")
            records = chunk.to_dict(orient="records")
            session.execute_write(import_batch, records)


try:
    run_import(driver, csv_file_path)
    print("Proses impor berhasil.")
finally:
    driver.close()

Membaca file data dari: ../UTS/Data/events.csv
Memulai proses impor data ke Neo4j...


Mengimpor Batch Data: 100%|██████████| 400/400 [02:49<00:00,  2.36it/s]

Proses impor berhasil.


In [7]:

FORCE_RESET = False

records, _, _ = driver.execute_query("MATCH (n) RETURN count(n) AS total_nodes", database_="neo4j")
current_nodes = records[0]["total_nodes"]

if current_nodes > 10 and not FORCE_RESET:
    print(f"Database contains full dataset ({current_nodes:,} nodes). Reset skipped to protect data.")
else:
    print("Resetting database...")
    while True:
        records, _, _ = driver.execute_query(
            "MATCH (n) WITH n LIMIT 10000 DETACH DELETE n RETURN count(n) AS deleted;",
            database_="neo4j"
        )
        if records[0]["deleted"] == 0:
            break
    print("Database reset complete.")


C:\Users\Hype\AppData\Local\Temp\ipykernel_11088\1900992344.py:3: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, _, _ = driver.execute_query("MATCH (n) RETURN count(n) AS total_nodes", database_="neo4j")


Database contains full dataset (1,248,002 nodes). Reset skipped to protect data.
